# Quantized messages

The LLRs the turbo loop passes between the equalizer and the decoder, held to `b` bits with a power-of-two step, `L0 = 3`, MAP in both BCJRs, then the chosen widths again with max-log-MAP. Simulated coded BER against what the EXIT chart predicts from the same quantizers. Reads the records under `results/quant`; BER points resting on fewer than 30 error events are dropped.

The prediction runs the receiver's 8 passes on the two EXIT characteristics and reads the BER off the decoder's measured BER at the input it ends on.

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
BER = ROOT / "results" / "quant" / "ber"
EXIT = ROOT / "results" / "quant" / "exit"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

FLOOR = 30       # minimum error events for a BER point to be drawn
TARGET = 1e-4    # where the losses are read
PASSES = 8       # 7 turbo iterations
EB_N0_DB = 5.0   # Eb/N0 of the EXIT panel
FAMILIES = ("balanced", "unbalanced")

# (bits, step) of every quantizer simulated: per width, three power-of-two steps.
QUANTIZERS = [(2, 8), (2, 4), (2, 2), (3, 4), (3, 2), (3, 1), (4, 2), (4, 1), (4, 0.5),
              (5, 1), (5, 0.5), (5, 0.25), (6, 1), (6, 0.5), (6, 0.25)]
WIDTHS = (2, 3, 4, 5, 6)

def suffix(q):
    return "" if q is None else f"_q{q[0]}_step{q[1]:g}"

# `algo` is the record suffix of the BCJR: "" for MAP, "_maxlog" for max-log-MAP.
def curve(fam, q=None, algo=""):
    """Eb/N0, BER and the 95% interval of one simulated curve, ordered and filtered."""
    pts = []
    for f in (BER / f"nsm_L3_{fam}_conv_K3_7iters{algo}{suffix(q)}").glob("snr_*dB.json"):
        r = json.loads(f.read_text())
        if r["ers_cnt"] >= FLOOR:
            pts.append((r["eb_no_db"], r["ber"], r["ber_lo95"], r["ber_ub95"]))
    pts.sort()
    return [np.array(c) for c in zip(*pts)]

def crossing(eb, ber, target=TARGET):
    """Eb/N0 where the curve crosses `target`, interpolated in log BER."""
    return float(np.interp(-np.log10(target), -np.log10(ber), eb))

def equalizer(fam, q=None, algo=""):
    """Equalizer characteristic per Eb/N0, as (MI in, MI out) of the messages."""
    d = json.loads((EXIT / f"nsm_L3_{fam}{algo}{suffix(q)}.json").read_text())
    out = {}
    for key, r in d["results"].items():
        x = np.array(r["IA_q"] if q else d["IA"])
        y = np.array(r["IE_q"] if q else r["IE_avg"])
        o = np.argsort(x, kind="stable")
        out[float(key)] = (x[o], y[o])
    return dict(sorted(out.items()))

def decoder(q=None, algo=""):
    """Decoder characteristic and the BER of its decisions, against MI in."""
    d = json.loads((EXIT / f"coder_K3{algo}{suffix(q)}.json").read_text())
    x = np.array(d["IA_q"] if q else d["IA"])
    y = np.array(d["IE_q"] if q else d["IE_avg"])
    o = np.argsort(x, kind="stable")
    return x[o], y[o], np.array(d["BER"])[o]

def predicted(fam, q=None, algo=""):
    """Eb/N0 and the BER the EXIT trajectory ends on after PASSES passes."""
    eq, (dx, dy, dber) = equalizer(fam, q, algo), decoder(q, algo)
    snr, ber = [], []
    for s, (ex, ey) in eq.items():
        x = 0.0
        for _ in range(PASSES - 1):
            x = float(np.interp(np.interp(x, ex, ey), dx, dy))
        y = float(np.interp(x, ex, ey))   # decoder input on the last pass
        snr.append(s)
        ber.append(10 ** np.interp(y, dx, np.log10(np.maximum(dber, 1e-9))))
    return np.array(snr), np.array(ber)

In [ ]:
# Eb/N0 at 1e-4, simulated and predicted, and the loss of each quantizer.
sim = {fam: crossing(*curve(fam)[:2]) for fam in FAMILIES}
pre = {fam: crossing(*predicted(fam)) for fam in FAMILIES}
loss_sim = {(fam, q): crossing(*curve(fam, q)[:2]) - sim[fam] for fam in FAMILIES for q in QUANTIZERS}
loss_pre = {(fam, q): crossing(*predicted(fam, q)) - pre[fam] for fam in FAMILIES for q in QUANTIZERS}

def best(loss, b):
    """Step with the lowest loss at width b, averaged over the two families."""
    return min((q for q in QUANTIZERS if q[0] == b), key=lambda q: np.mean([loss[f, q] for f in FAMILIES]))

BEST = {b: best(loss_pre, b) for b in WIDTHS}   # chosen from EXIT alone

for fam in FAMILIES:
    print(f"{fam}: float at 1e-4 simulated {sim[fam]:.2f} dB, predicted {pre[fam]:.2f} dB")
print("\n bits  step  clip  | balanced sim / EXIT | unbalanced sim / EXIT   (loss in dB)")
for q in QUANTIZERS:
    mark = "*" if BEST[q[0]] == q else " "
    clip = q[1] * (2 ** (q[0] - 1) - 1)
    print(f"{mark}{q[0]:4d} {q[1]:5g} {clip:5g}  |  {loss_sim['balanced', q]:+.2f} / {loss_pre['balanced', q]:+.2f}"
          f"      |  {loss_sim['unbalanced', q]:+.2f} / {loss_pre['unbalanced', q]:+.2f}")
print("\nstep EXIT picks / step BER picks:", {b: (BEST[b][1], best(loss_sim, b)[1]) for b in WIDTHS})

In [ ]:
FAMILY_COLOR = {"balanced": "#2a78d6", "unbalanced": "#eb6834"}
WIDTH_COLOR = {2: "#86b6ef", 3: "#5598e7", 4: "#2a78d6", 5: "#1c5cab", 6: "#104281"}
WIDTH_MARKER = {2: "o", 3: "s", 4: "^", 5: "D", 6: "v"}
FLOAT = "#000000"

fig, axes = plt.subplots(2, 2, figsize=(7.6, 7.0))

ax = axes[0, 0]
for fam in FAMILIES:
    qs = [BEST[b] for b in WIDTHS]
    ax.plot(WIDTHS, [loss_sim[fam, q] for q in qs], color=FAMILY_COLOR[fam], marker="o",
            markersize=4, label=f"{fam}, simulated")
    ax.plot(WIDTHS, [loss_pre[fam, q] for q in qs], color=FAMILY_COLOR[fam], marker="o",
            markersize=4, ls="--", mfc="white", label=f"{fam}, EXIT")
ax.axhline(0, color="0.5", lw=0.6)
ax.set_xticks(WIDTHS, [f"{b}\nstep {BEST[b][1]:g}" for b in WIDTHS])
ax.set(xlabel="bits per message", ylabel="loss at BER $10^{-4}$ (dB)",
       title="Loss against bit width, EXIT's step per width")
ax.legend(loc="upper right")

ax = axes[0, 1]
lim = 1.05 * max(max(loss_sim.values()), max(loss_pre.values()))
ax.plot([-0.1, lim], [-0.1, lim], color="0.5", lw=0.6)
for (fam, q), ls in loss_sim.items():
    ax.plot(ls, loss_pre[fam, q], ls="none", marker=WIDTH_MARKER[q[0]], markersize=4.5,
            color=FAMILY_COLOR[fam], mec="white", mew=0.6)
for fam in FAMILIES:
    ax.plot([], [], ls="none", marker="o", color=FAMILY_COLOR[fam], label=fam)
for b in WIDTHS:
    ax.plot([], [], ls="none", marker=WIDTH_MARKER[b], color="0.45", label=f"{b} bits")
ax.set(xlim=(-0.1, lim), ylim=(-0.1, lim), xlabel="simulated loss (dB)", ylabel="EXIT-predicted loss (dB)",
       title="All 15 quantizers, three steps per width")
ax.legend(loc="upper left", ncol=2)

ax = axes[1, 0]
fam = "balanced"
for q, color, label in [(None, FLOAT, "float")] + [(BEST[b], WIDTH_COLOR[b], f"{b} bits") for b in (4, 3, 2)]:
    eq, (dx, dy, _) = equalizer(fam, q), decoder(q)
    s = min(eq, key=lambda v: abs(v - EB_N0_DB))
    ax.plot(*eq[s], color=color, label=f"{label}, equalizer")
    ax.plot(dy, dx, color=color, ls="--", lw=1.0)
ax.plot([], [], color="0.45", ls="--", lw=1.0, label="decoder, mirrored")
ax.plot([0, 1], [0, 1], ls=":", lw=0.7, color="0.6")
ax.set(xlim=(0, 1), ylim=(0, 1), title=f"{fam}, EXIT at {EB_N0_DB:g} dB",
       xlabel=r"$I_A^{\mathrm{modem}} = I_E^{\mathrm{dec}}$", ylabel=r"$I_E^{\mathrm{modem}} = I_A^{\mathrm{dec}}$")
ax.legend(loc="lower right")

ax = axes[1, 1]
eb, ber, lo, hi = curve(fam)
ax.errorbar(eb, ber, yerr=[ber - lo, hi - ber], color=FLOAT, markersize=3, marker=".",
            elinewidth=0.7, capsize=1.5, label=f"float, {sim[fam]:.2f} dB")
for b in WIDTHS:
    eb, ber, lo, hi = curve(fam, BEST[b])
    ax.errorbar(eb, ber, yerr=[ber - lo, hi - ber], color=WIDTH_COLOR[b], marker=WIDTH_MARKER[b],
                markersize=3, elinewidth=0.7, capsize=1.5,
                label=f"{b} bits, {sim[fam] + loss_sim[fam, BEST[b]]:.2f} dB")
ax.axhline(TARGET, color="0.5", lw=0.6)
ax.set(yscale="log", xlabel="$E_b/N_0$ (dB)", ylabel="BER", title=f"{fam}, coded, 7 iterations",
       ylim=(3e-6, 0.5))
ax.legend(loc="lower left", title="at $10^{-4}$", title_fontsize=6.5)

fig.tight_layout()
fig.savefig(FIGURES / "llr_quant.png")

In [ ]:
# The chosen quantizers again with max-log-MAP in both BCJRs, the usual choice in hardware.
ML = "_maxlog"
ml_sim = {fam: crossing(*curve(fam, algo=ML)[:2]) for fam in FAMILIES}
ml_pre = {fam: crossing(*predicted(fam, algo=ML)) for fam in FAMILIES}
for fam in FAMILIES:
    print(f"{fam}: max-log float at 1e-4 simulated {ml_sim[fam]:.2f} dB, predicted {ml_pre[fam]:.2f} dB")
print("\n bits  step  | balanced sim / EXIT | unbalanced sim / EXIT   (loss against max-log float, dB)")
for b in WIDTHS:
    q = BEST[b]
    row = [f"{crossing(*curve(f, q, ML)[:2]) - ml_sim[f]:+.2f} / {crossing(*predicted(f, q, ML)) - ml_pre[f]:+.2f}"
           for f in FAMILIES]
    print(f"{b:5d} {q[1]:5g}  |  {row[0]}      |  {row[1]}")